# 1. Ask the database a precise question

The desk's prices live in a database now. Your task: **retrieve the right observations for a stock report and explain what each row means**.
This notebook starts independently; download the bCourses database into this lecture's `data/` folder before class.

You will use SQL to choose data and pandas to inspect and analyze it. SQL expresses many of the same table operations you have used in pandas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path
from IPython.display import display
%matplotlib inline
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
lesson_dir = course_root / 'Lectures/Lecture 4'
plt.style.use('seaborn-v0_8-whitegrid')

## 1. What must be true before calculating a return?
**Recall:** Why do we need **ticker, date order, and valid prices** before calculating returns?
A database contains tables. A connection lets Python send queries to it. The next cell checks that the database file exists and opens a read-only connection.

In [ ]:
database = lesson_dir / 'data/data.db'
if not database.is_file():
    raise FileNotFoundError(f'Download data.db from bCourses and place it at {database}')
conn = sqlite3.connect(database.resolve().as_uri() + '?mode=ro', uri=True)
display(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn))

In [ ]:
preview = pd.read_sql_query('SELECT ts, ticker, close, volume FROM ohlc LIMIT 5', conn)
display(preview)
display(preview.dtypes)

**Discuss:** one row represents one ticker on one date. That pair is our intended key.
`ts` is the date column. `ohlc` is the supplied price table; `ohlc_hw` contains deliberately dirty homework data.
`LIMIT 5` is a preview, not a time-order guarantee. We will request an explicit order for analysis.

File reading follows the same inspect-after-load habit. CSV options and nested JSON are in the optional [ingestion reference](reference/2_data_ingestion_and_sql.ipynb).

## 2. Translate a question into a query
Question: **“Show AAPL closing prices in the first half of 2024, oldest first.”**

| SQL | Familiar pandas idea |
|---|---|
| `SELECT ts, close` | choose columns |
| `FROM ohlc` | choose the table |
| `WHERE ...` | a row mask |
| `ORDER BY ts` | `sort_values('ts')` |

**Predict:** Which ticker should appear? What is the earliest possible date?

In [ ]:
query = """
SELECT ts, ticker, close, volume
FROM ohlc
WHERE ticker = 'AAPL'
  AND ts >= '2024-01-01' AND ts <= '2024-06-30'
ORDER BY ts
"""
aapl = pd.read_sql_query(query, conn, parse_dates=['ts'])
display(aapl.head())
print('Rows:', len(aapl), '| First date:', aapl['ts'].min(), '| Last date:', aapl['ts'].max())

### Make it reusable with a value parameter
The `?` placeholders accept values from `params`; the SQL structure stays the same.
Use this instead of building SQL by pasting user values into a string.
**Change one thing:** switch `selected_ticker` to MSFT, run the cell, explain what changed, then restore AAPL.

In [ ]:
selected_ticker = 'AAPL'
start_date, end_date = '2024-01-01', '2024-06-30'
price_query = """
SELECT ts, ticker, close, volume FROM ohlc
WHERE ticker = ? AND ts BETWEEN ? AND ? ORDER BY ts
"""
selected = pd.read_sql_query(price_query, conn, params=(selected_ticker, start_date, end_date), parse_dates=['ts'])
selected.head()

## 3. Worked example: verify AAPL closes above $180
Add a price condition to the query, then express the same condition in pandas. Matching rows, columns, values, and order gives us a direct check on the SQL.

In [ ]:
example_query = """
SELECT ts, ticker, close FROM ohlc
WHERE ticker = ? AND ts BETWEEN ? AND ? AND close > ? ORDER BY ts
"""
example_query_result = pd.read_sql_query(example_query, conn, params=('AAPL', start_date, end_date, 180), parse_dates=['ts'])
example_expected_filter = aapl.loc[aapl['close'] > 180, ['ts', 'ticker', 'close']].reset_index(drop=True)
pd.testing.assert_frame_equal(example_query_result, example_expected_filter)
display(example_query_result.head())

## 4. Group and join without changing the question
`GROUP BY ticker` makes one summary row per ticker. It is the database equivalent of `groupby`.
**Predict:** how many rows should the next query return? Is average share price a performance measure?

In [ ]:
sql_summary = pd.read_sql_query("""
SELECT ticker, COUNT(*) AS observations, AVG(close) AS mean_close, AVG(volume) AS mean_volume
FROM ohlc
WHERE ticker IN ('AAPL', 'MSFT', 'NVDA') AND ts BETWEEN ? AND ?
GROUP BY ticker ORDER BY ticker
""", conn, params=(start_date, end_date))
display(sql_summary)
assert np.isclose(sql_summary.set_index('ticker').loc['AAPL', 'mean_close'], aapl['close'].mean())

### The complete join key matters
A market-cap observation has both a ticker and a date. Joining only on ticker could multiply each price row by many dated cap rows.
A **left join** keeps the price dates and leaves missing values when there is no matching cap on that date.
**Predict:** should this left join create more AAPL price rows? What would an inner join discard?

In [ ]:
joined = pd.read_sql_query("""
SELECT o.ts, o.ticker, o.close, m.market_capitalization / 1e9 AS market_cap_bn
FROM ohlc AS o
LEFT JOIN market_caps AS m ON o.ticker = m.ticker AND o.ts = m.ts
WHERE o.ticker = ? AND o.ts BETWEEN ? AND ?
ORDER BY o.ts
""", conn, params=('AAPL', start_date, end_date), parse_dates=['ts'])
print('Before:', len(aapl), '| After:', len(joined))
assert len(joined) == len(aapl), 'Investigate repeated keys in the lookup.'
display(joined.loc[joined['market_cap_bn'].notna()].head())
print('Dates with no matching cap:', joined['market_cap_bn'].isna().sum())

## 5. Worked example: check an NVDA extract
Check the requested ticker and period, chronological order, unique keys, and finite positive closes. These checks describe the rows we retrieved; they do not establish that every expected trading date is present.

In [ ]:
example_extract = pd.read_sql_query(price_query, conn, params=('NVDA', start_date, end_date), parse_dates=['ts'])
assert not example_extract.empty and example_extract['ticker'].eq('NVDA').all()
assert example_extract['ts'].between(pd.Timestamp(start_date), pd.Timestamp(end_date)).all()
assert example_extract['ts'].is_monotonic_increasing
assert not example_extract.duplicated(['ticker', 'ts']).any()
assert np.isfinite(example_extract['close']).all() and example_extract['close'].gt(0).all()
display(example_extract.head())
print('NVDA rows passed the checks; calendar completeness and vendor accuracy still need separate evidence.')
# COUNT(column) excludes nulls; COUNT(*) includes every row.
example_missing_volume = pd.read_sql_query("""
SELECT ticker, COUNT(*) AS rows, COUNT(*) - COUNT(volume) AS missing_volumes
FROM ohlc_hw GROUP BY ticker ORDER BY ticker
""", conn)
display(example_missing_volume.head())

### Bridge to the report and homework
The quality checks next module will also inspect raw open/high/low/close and volume.
Missing **rows** differ from null **cells**: calendar coverage requires an expected exchange calendar and the asset's active history. Weekdays alone include exchange holidays.
A statistical outlier deserves review; an OHLC inconsistency needs investigation before calculating risk.

**Exit:** explain the role of SELECT, WHERE, ORDER BY, and the two-column join key without looking at the examples.
Next: [Turn prices into a report you understand](2_stock_analysis_template.ipynb).

## Optional: write a table, read it back
This provided example writes a small summary to a separate output database. The source remains read-only.
Use this pattern for homework's cleaned table. The longer [ingestion notebook](reference/2_data_ingestion_and_sql.ipynb) covers CSV/JSON, cursors, CTEs, and more query variants.

In [ ]:
output_dir = lesson_dir / 'outputs'
output_dir.mkdir(exist_ok=True)
demo_conn = sqlite3.connect(output_dir / 'cleaning_demo.db')
sql_summary.to_sql('ticker_summary', demo_conn, if_exists='replace', index=False)
roundtrip = pd.read_sql_query('SELECT * FROM ticker_summary ORDER BY ticker', demo_conn)
pd.testing.assert_frame_equal(roundtrip, sql_summary)
demo_conn.close()

## Exercises
Use the tables and functions above to complete these tasks. Replace the placeholders with your code, then run the checks.

### Your turn E: write and verify a filter
Retrieve `ts`, `ticker`, and `close` for **MSFT** in the first half of 2024 **with close above 400**, ordered by date.
Put your SQL string in `your_query` and its result in `your_query_result`.
**Hint:** reuse SELECT/FROM/WHERE/ORDER BY and add `AND close > 400`.
**Stretch:** use a price placeholder as well as a ticker placeholder.

Check your query against pandas, then change one condition and explain the result.

In [ ]:
your_query = None
your_query_result = None
# your_query = """SELECT ... FROM ... WHERE ... ORDER BY ..."""
# your_query_result = pd.read_sql_query(your_query, conn, parse_dates=['ts'])

In [ ]:
msft = pd.read_sql_query(price_query, conn, params=('MSFT', start_date, end_date), parse_dates=['ts'])
expected_filter = msft.loc[msft['close'] > 400, ['ts', 'ticker', 'close']].reset_index(drop=True)
if your_query_result is None:
    print('Try E, then compare the SQL result with this pandas selection.')
else:
    pd.testing.assert_frame_equal(your_query_result.reset_index(drop=True), expected_filter)
    print('E passed: SQL and pandas give the same rows, columns, values, and order.')
display(expected_filter.head())

### Your turn: prepare a JPM report input
Request **JPM** observations for the first half of 2024 using `price_query`. Store them in `your_extract`.
Check ticker, date order, duplicate `(ticker, ts)` keys, and missing/nonpositive closes.
Write `your_extract_note`: distinguish **the checks you ran** from **what you have not established**.

**Hint:** `.is_monotonic_increasing`, `.duplicated(['ticker', 'ts'])`, `.isna()`, and a comparison mask.
**Stretch:** count missing closes per ticker in `ohlc_hw` using `COUNT(*) - COUNT(close)`.
Check the extract and explain what these checks cannot prove.

In [ ]:
your_extract = None
your_extract_note = ''

In [ ]:
if your_extract is None:
    print('Prepare the JPM extract and a short quality note.')
else:
    assert not your_extract.empty and your_extract['ticker'].eq('JPM').all()
    assert your_extract['ts'].between(pd.Timestamp(start_date), pd.Timestamp(end_date)).all()
    assert your_extract['ts'].is_monotonic_increasing
    assert not your_extract.duplicated(['ticker', 'ts']).any()
    assert np.isfinite(your_extract['close']).all() and your_extract['close'].gt(0).all()
    assert your_extract_note.strip()
    print('Extract checks passed. These checks do not prove complete trading-calendar coverage.')